# Instagram Scraper
Date range: 2026-01-20 → 2026-05-18


Downloads images + videos + creates Excel guide + Gemini descriptions

*Fully updated for latest apify-client (uses .default_dataset_id)*

## 🛠️ Step 1: Environment Setup
Install the required libraries to handle API requests, AI processing, and data management.

### Installation Command
To install the necessary dependencies, run the following command in a code cell:

```bash
!pip install -q apify-client google-generativeai openpyxl pandas requests tqdm pillow
```

## 🔑 Step 2: Authentication Test
Use this section to verify your Apify token before starting the full process.

# **Run the code before you run with main code run to see if Appify token works.**

In [ ]:
from apify_client import ApifyClient

APIFY_TOKEN = "paste your APIFY PERSONAL KEY HERE"   # paste your real token

client = ApifyClient(APIFY_TOKEN)
user = client.user().get()

print("✅ Token works!")
print("User ID / username:", getattr(user, "username", None) or getattr(user, "id", None))
print("Full object type:", type(user))

## 🚀 Step 3: Main Instagram Scraper & Gemini AI
This is the core engine that downloads media and generates AI descriptions.

In [ ]:
# ============================================================
# Instagram Scraper
# Date range: 2026-01-20 → 2026-05-18
# Downloads images + videos + creates Excel guide + Gemini descriptions
# Fully updated for latest apify-client (uses .default_dataset_id)
# ============================================================

# 1. Install packages
!pip install -q apify-client google-generativeai openpyxl pandas requests tqdm pillow

import os
import json
import time
import requests
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from tqdm.notebook import tqdm
from apify_client import ApifyClient
import google.generativeai as genai
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows

# ----------------------------------------------------------
# 2. FILL IN YOUR KEYS HERE
# ----------------------------------------------------------
APIFY_TOKEN    = "PASTE YOUR APIFY PERSONAL API KEY HERE"          # ← paste your working token
GEMINI_API_KEY = "PASTER YOUR GEMINI API KEY HERE"

# ----------------------------------------------------------
# 3. Settings
# ----------------------------------------------------------
USERNAME     = "username" # instagram username .eg. @vscode = vscode ->username
PROFILE_URL  = f"https://www.instagram.com/{USERNAME}/"

START_DATE   = "2026-01-20"   # inclusive/ you can adjust it or remove this if you dont want any start and end date.
END_DATE     = "2026-05-18"   # inclusive/ you can adjust it or remove this if you dont want any start and end date.

RESULTS_LIMIT = 100
INCLUDE_REELS = True
DOWNLOAD_MEDIA = True
GENERATE_DESCRIPTIONS = True

# Folders
BASE_DIR   = Path("/content/instagram_scrape")
MEDIA_DIR  = BASE_DIR / "media"
POSTS_DIR  = MEDIA_DIR / "posts"
VIDEOS_DIR = MEDIA_DIR / "videos"
IMAGES_DIR = MEDIA_DIR / "images"

for folder in [POSTS_DIR, VIDEOS_DIR, IMAGES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# 4. Initialize clients
# ----------------------------------------------------------
apify = ApifyClient(APIFY_TOKEN)
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-1.5-flash")

print("✅ Clients ready")

# ----------------------------------------------------------
# 5. Run Apify Instagram Scraper
# ----------------------------------------------------------
print(f"\n🚀 Scraping posts newer than {START_DATE}...")

run_input = {
    "directUrls": [PROFILE_URL],
    "resultsType": "posts",
    "resultsLimit": RESULTS_LIMIT,
    "onlyPostsNewerThan": START_DATE,
    "addParentData": False,
}

run = apify.actor("apify/instagram-scraper").call(run_input=run_input)

# ✅ Correct way for latest apify-client
items = list(apify.dataset(run.default_dataset_id).iterate_items())
print(f"✅ Posts run finished → {len(items)} items")

# Also scrape reels
if INCLUDE_REELS:
    print("🎬 Scraping reels...")
    reels_input = {
        "directUrls": [PROFILE_URL],
        "resultsType": "reels",
        "resultsLimit": RESULTS_LIMIT,
        "onlyPostsNewerThan": START_DATE,
    }
    reels_run = apify.actor("apify/instagram-scraper").call(run_input=reels_input)
    reels_items = list(apify.dataset(reels_run.default_dataset_id).iterate_items())

    existing = {item.get("shortCode") for item in items}
    for r in reels_items:
        if r.get("shortCode") not in existing:
            items.append(r)
    print(f"✅ After merging reels → {len(items)} unique items")

# ----------------------------------------------------------
# 6. Filter to exact date range
# ----------------------------------------------------------
def parse_timestamp(ts):
    if not ts:
        return None
    try:
        if isinstance(ts, (int, float)):
            return datetime.fromtimestamp(ts, tz=timezone.utc)
        ts = str(ts).replace("Z", "+00:00")
        return datetime.fromisoformat(ts).astimezone(timezone.utc)
    except:
        return None

start_dt = datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc)
end_dt   = datetime.fromisoformat(END_DATE).replace(hour=23, minute=59, second=59, tzinfo=timezone.utc)

filtered = []
for post in items:
    ts = parse_timestamp(post.get("timestamp"))
    if ts and start_dt <= ts <= end_dt:
        filtered.append(post)

items = filtered
print(f"📅 After date filter ({START_DATE} → {END_DATE}): {len(items)} posts kept")

# Save raw data
with open(BASE_DIR / "raw_filtered_data.json", "w", encoding="utf-8") as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------
# 7. Download helper
# ----------------------------------------------------------
def download_file(url, save_path):
    if not url:
        return False
    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
        r = requests.get(url, headers=headers, stream=True, timeout=60)
        r.raise_for_status()
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        return True
    except Exception as e:
        print(f"   ⚠️ Download failed: {e}")
        return False

# ----------------------------------------------------------
# 8. Process posts → download media + Gemini description
# ----------------------------------------------------------
rows = []
print("\n📥 Downloading media and generating descriptions...")

for idx, post in enumerate(tqdm(items, desc="Processing posts")):
    shortcode = post.get("shortCode") or post.get("id") or f"post_{idx}"
    post_url  = post.get("url") or f"https://www.instagram.com/p/{shortcode}/"
    caption   = post.get("caption") or ""
    timestamp = post.get("timestamp") or ""
    likes     = post.get("likesCount")
    comments  = post.get("commentsCount")
    post_type = post.get("type") or post.get("productType") or "Unknown"

    post_folder = POSTS_DIR / shortcode
    post_folder.mkdir(exist_ok=True)

    media_files = []
    media_index = 0

    # Main media
    video_url   = post.get("videoUrl")
    display_url = post.get("displayUrl")

    if video_url and DOWNLOAD_MEDIA:
        fname = f"{shortcode}_video_{media_index:02d}.mp4"
        path = post_folder / fname
        if download_file(video_url, path):
            media_files.append((fname, "video", video_url))
            (VIDEOS_DIR / fname).write_bytes(path.read_bytes())
        media_index += 1
    elif display_url and DOWNLOAD_MEDIA:
        fname = f"{shortcode}_img_{media_index:02d}.jpg"
        path = post_folder / fname
        if download_file(display_url, path):
            media_files.append((fname, "image", display_url))
            (IMAGES_DIR / fname).write_bytes(path.read_bytes())
        media_index += 1

    # Carousel children
    for child in (post.get("childPosts") or post.get("sidecarChildren") or []):
        c_video   = child.get("videoUrl")
        c_display = child.get("displayUrl")

        if c_video and DOWNLOAD_MEDIA:
            fname = f"{shortcode}_video_{media_index:02d}.mp4"
            path = post_folder / fname
            if download_file(c_video, path):
                media_files.append((fname, "video", c_video))
                (VIDEOS_DIR / fname).write_bytes(path.read_bytes())
            media_index += 1
        elif c_display and DOWNLOAD_MEDIA:
            fname = f"{shortcode}_img_{media_index:02d}.jpg"
            path = post_folder / fname
            if download_file(c_display, path):
                media_files.append((fname, "image", c_display))
                (IMAGES_DIR / fname).write_bytes(path.read_bytes())
            media_index += 1

    # Gemini description
    description = ""
    if GENERATE_DESCRIPTIONS:
        try:
            prompt = f"""Write a short 1-3 sentence description of this Instagram post so it can be easily identified later.
Focus on the main visual content and topic. Do not invent details.

Caption: {caption[:1200] if caption else "(no caption)"}
Post type: {post_type}
Number of media items: {len(media_files)}
"""
            response = gemini_model.generate_content(prompt)
            description = response.text.strip()
            time.sleep(1.2)
        except Exception as e:
            description = f"[Gemini error: {e}]"

    # Excel rows
    if media_files:
        for fname, mtype, orig_url in media_files:
            rows.append({
                "post_shortcode": shortcode,
                "post_url": post_url,
                "post_type": post_type,
                "timestamp": timestamp,
                "likes": likes,
                "comments": comments,
                "caption": (caption[:400] + "...") if len(caption) > 400 else caption,
                "gemini_description": description,
                "media_filename": fname,
                "media_type": mtype,
                "local_folder": str(post_folder.relative_to(BASE_DIR)),
                "original_media_url": (orig_url[:180] + "...") if orig_url else "",
            })
    else:
        rows.append({
            "post_shortcode": shortcode,
            "post_url": post_url,
            "post_type": post_type,
            "timestamp": timestamp,
            "likes": likes,
            "comments": comments,
            "caption": (caption[:400] + "...") if len(caption) > 400 else caption,
            "gemini_description": description,
            "media_filename": "",
            "media_type": "",
            "local_folder": str(post_folder.relative_to(BASE_DIR)),
            "original_media_url": "",
        })

# ----------------------------------------------------------
# 9. Create Excel guide
# ----------------------------------------------------------
print("\n📊 Creating Excel guide...")

df = pd.DataFrame(rows)
excel_path = BASE_DIR / f"{USERNAME}_{START_DATE}_to_{END_DATE}_media_guide.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Media Guide"

header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
header_font = Font(bold=True, color="FFFFFF")

for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
    for c_idx, value in enumerate(row, 1):
        cell = ws.cell(row=r_idx, column=c_idx, value=value)
        if r_idx == 1:
            cell.fill = header_fill
            cell.font = header_font
        cell.alignment = Alignment(wrap_text=True, vertical="top")

widths = [16, 42, 12, 22, 8, 8, 45, 50, 32, 10, 25, 35]
for i, width in enumerate(widths, 1):
    ws.column_dimensions[chr(64 + i)].width = width

wb.save(excel_path)

csv_path = BASE_DIR / f"{USERNAME}_{START_DATE}_to_{END_DATE}_media_guide.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

# ----------------------------------------------------------
# 10. Final summary
# ----------------------------------------------------------
print("\n" + "="*60)
print("✅ DONE")
print(f"Posts kept in date range : {len(items)}")
print(f"Media files downloaded   : {sum(1 for r in rows if r['media_filename'])}")
print(f"Excel guide              : {excel_path}")
print(f"All files location       : {BASE_DIR}")
print("="*60)

print("\nTo download everything, run this next:")
print("!zip -r "USERNAME"_scrape.zip /content/instagram_scrape")

To download everything, run this next:

## 📦 Step 4: Export Results
Compress the downloaded media and the Excel guide into a single file for download.

In [ ]:
!zip -r justcutitgh_scrape.zip /content/instagram_scrape